# arXiv 논문 인용 정보 수집 (Semantic Scholar API)

`data/cleaned/*.jsonl`(arXiv API로 수집한 정제 데이터)에 들어 있는 모든 논문 `id`를 대상으로,
[Semantic Scholar Graph API](https://api.semanticscholar.org/api-docs/graph) 배치 조회
엔드포인트(`POST /graph/v1/paper/batch`)를 이용해 인용 정보를 수집하고
`data/citations/arxiv_citations_part{n}.jsonl`로 저장합니다.

## 무엇을 수집하나
- `citation_count` / `reference_count` / `influential_citation_count`: 논문 단위 요약 지표.
- `citations`: 이 논문을 인용한 논문 목록(일부 필드만: Semantic Scholar id, arXiv id, 제목).
- `references`: 이 논문이 인용한 논문 목록(같은 필드).
- `found`: Semantic Scholar 색인에서 찾았는지 여부. arXiv에 막 올라온 논문은 아직 색인되지
  않아 `false`일 수 있습니다.

arXiv id는 버전 접미사(`v1`, `v2`, ...)를 뺀 형태로 사용합니다. Semantic Scholar는 버전을
따로 구분하지 않고 `ARXIV:<id>` 하나로 논문을 식별하기 때문입니다. 같은 논문이 여러
`data/cleaned` 파일에 걸쳐 있으면 가장 높은 버전 하나만 대상으로 삼습니다(다른 노트북들과
동일한 규칙).

## 재시작(에러 대비 상태 저장)
`arxiv_computer_science_recent_3months_to_jsonl.ipynb`와 같은 방식입니다:
- 수집 결과는 배치(최대 500편)를 처리할 때마다 즉시 `data/citations/*.jsonl`에 flush됩니다.
- 재실행 시 기존 출력 파일들을 먼저 읽어 이미 수집된 `arxiv_id` 집합을 만들고, 남은 논문만
  다시 조회합니다. API 에러로 중간에 멈춰도 이미 받은 배치는 디스크에 남아 있으므로 셀을
  다시 실행하면 이어서 진행됩니다.
- `arxiv_citations_state.json`은 이번 대상 논문 목록의 서명(해시)과 마지막 진행 상황을
  기록하는 참고용 메타데이터입니다. 대상 목록이 달라져도(예: 새 논문 추가) 기존에 모은
  인용 정보를 지우지 않고 늘어난 분량만 추가로 수집합니다(날짜 구간을 다시 긋는 원본 수집
  노트북과 달리, 여기서는 대상이 계속 누적되기만 하므로 파괴적 초기화를 하지 않습니다).

## API 키 없이 쓸 때 (기본값)
Semantic Scholar는 API 키가 있으면 공식적으로 **1편/초**를 보장합니다(리뷰를 거쳐 더
높게 주는 경우도 있음). 키가 없으면 모든 비인증 사용자가 하나의 공용 한도를 나눠 쓰는데,
이 공용 한도는 엔드포인트마다 실제로 걸리는 정도가 달라서 공식 문서에 `/paper/batch`
전용 고정 수치가 없습니다. 실측 보고(2026-08-26,
[CASRAI 가이드](https://casrai.org/guides/semantic-scholar-api))에 따르면 `/paper/batch`는
간격 없이 연속 호출했을 때 6번 중 1번만 성공할 정도로 다른 엔드포인트보다 먼저 막힙니다.

그래서 이 노트북은 **키가 없으면 요청 사이 간격을 5초로 늘리고**(키가 있으면 1초),
그래도 429가 나면 지수 백오프(최대 `MAX_RETRIES=6`회, 최대 대기 120초)로 재시도합니다.
공식적으로 문서화된 정확한 수치가 아니라 보수적으로 잡은 값이므로, 429가 자주 보이면
`REQUEST_INTERVAL_SECONDS`를 더 늘리고, 반대로 순조롭게 진행되면 줄여도 됩니다.
`.env`에 `SEMANTIC_SCHOLAR_API_KEY`를 넣으면([발급 안내](https://www.semanticscholar.org/product/api#api-key))
더 짧은 간격과 더 안정적인 처리량을 쓸 수 있습니다.

In [1]:
import hashlib
import http.client
import json
import os
import re
import ssl
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

# 저장소 루트와 notebooks/ 어느 위치에서 커널을 시작해도 동작합니다.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / "cleaned").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 폴더에서 커널을 시작하세요.")
load_dotenv(ROOT / ".env", override=False)

INPUT_DIR = ROOT / "data" / "ai"
OUTPUT_DIR = ROOT / "data" / "citations_ai"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX = "arxiv_citations"
CHUNK_SIZE = 5000
STATE_PATH = OUTPUT_DIR / f"{FILE_PREFIX}_state.json"

S2_API_URL = "https://api.semanticscholar.org/graph/v1/paper/batch"
S2_FIELDS = (
    "externalIds,title,citationCount,referenceCount,influentialCitationCount,"
    "references.paperId,references.externalIds,references.title,"
    "citations.paperId,citations.externalIds,citations.title"
)
S2_BATCH_SIZE = 500  # Semantic Scholar 배치 엔드포인트 1회 호출 최대 논문 수
S2_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "").strip() or None
# 키가 있으면 공식 문서대로 1편/초를 보장받습니다. 키가 없으면 모든 비인증 사용자가
# 하나의 공용 한도를 나눠 쓰고, 특히 /paper/batch는 문서화된 고정 수치가 없이도
# 실측상 다른 엔드포인트보다 훨씬 빨리 429가 나는 것으로 보고되어 있어(참고:
# https://casrai.org/guides/semantic-scholar-api) 더 넉넉한 간격을 둡니다.
REQUEST_INTERVAL_SECONDS = 1.0 if S2_API_KEY else 5.0
MAX_RETRIES = 6
BACKOFF_BASE_SECONDS = 5.0
BACKOFF_MAX_SECONDS = 120.0


def chunk_path(chunk_index):
    return OUTPUT_DIR / f"{FILE_PREFIX}_part{chunk_index}.jsonl"


print(f"입력 폴더: {INPUT_DIR}")
print(f"출력 폴더: {OUTPUT_DIR} ({FILE_PREFIX}_part{{n}}.jsonl, {CHUNK_SIZE}건/파일)")
print(f"Semantic Scholar API 키: {'있음' if S2_API_KEY else '없음 (.env에 SEMANTIC_SCHOLAR_API_KEY 추가 가능)'}")
print(f"요청 간격: {REQUEST_INTERVAL_SECONDS}초 / 배치당 최대 {S2_BATCH_SIZE}편")

입력 폴더: c:\Users\Playdata\Desktop\arxiv_graph_RAG\data\ai
출력 폴더: c:\Users\Playdata\Desktop\arxiv_graph_RAG\data\citations_ai (arxiv_citations_part{n}.jsonl, 5000건/파일)
Semantic Scholar API 키: 없음 (.env에 SEMANTIC_SCHOLAR_API_KEY 추가 가능)
요청 간격: 5.0초 / 배치당 최대 500편


## 1. `data/cleaned`에서 조회 대상 arXiv id 수집

각 파일의 `id`(예: `http://arxiv.org/abs/2609.11929v1`)에서 버전을 뺀 `2609.11929`만
남기고, 같은 논문이 여러 파일에 있으면 최고 버전만 남깁니다. 정렬된 순서로 고유 id 목록을
만들어야 재실행할 때마다 같은 순서가 나오고, 이미 처리한 논문을 안정적으로 건너뛸 수
있습니다.

In [2]:
ID_PATTERN = re.compile(r"https?://arxiv\.org/abs/(\d{4}\.\d{4,5})(?:v(\d+))?")


def collect_target_paper_ids(files):
    """각 파일의 id에서 버전을 뺀 arXiv id만 모으고, 같은 논문은 최고 버전 하나만 남깁니다."""
    versions = {}
    malformed = 0
    for path in files:
        with path.open(encoding="utf-8-sig") as handle:
            for line in handle:
                if not line.strip():
                    continue
                try:
                    raw = json.loads(line)
                except json.JSONDecodeError:
                    malformed += 1
                    continue
                raw_id = raw.get("id") if isinstance(raw, dict) else None
                match = ID_PATTERN.fullmatch((raw_id or "").strip())
                if not match:
                    malformed += 1
                    continue
                paper_id, version = match[1], int(match[2] or 0)
                if paper_id not in versions or version > versions[paper_id]:
                    versions[paper_id] = version
    if malformed:
        print(f"경고: id를 해석하지 못한 줄 {malformed}건은 건너뜁니다.")
    return sorted(versions)


FILES = sorted(INPUT_DIR.glob("*.jsonl"))
if not FILES:
    raise FileNotFoundError(f"{INPUT_DIR}에 jsonl 파일이 없습니다. 먼저 정제된 데이터를 준비하세요.")

target_ids = collect_target_paper_ids(FILES)
target_signature = hashlib.sha256("\n".join(target_ids).encode("utf-8")).hexdigest()
print(f"입력 파일: {len(FILES)}개 / 고유 논문 id: {len(target_ids)}개")
print("예시:", target_ids[:3])

입력 파일: 2개 / 고유 논문 id: 6467개
예시: ['1304.3111', '2002.11508', '2006.04156']


## 2. Semantic Scholar 배치 조회 함수

`POST /graph/v1/paper/batch`에 `ids: ["ARXIV:<id>", ...]`(최대 500개)를 보내면, 요청한
순서 그대로 결과 배열을 돌려줍니다. Semantic Scholar에 없는 논문은 그 자리에 `null`이
들어오므로 `found=false`로 기록합니다.

`citations`/`references`는 전체 필드 대신 `paperId`/`externalIds`/`title`만 받아 응답
크기를 줄였습니다. 최근 3개월 이내 논문이 대상이라 `citations`(피인용) 목록은 대체로
짧고, `references`(참고문헌)는 논문 제출 시점에 고정된 값이라 무한정 커지지 않습니다.

429(요청 과다)·5xx 오류와 네트워크 오류는 지수 백오프로 재시도하고, 그래도 실패하면
예외를 그대로 올립니다. 이미 flush된 배치는 디스크에 남아 있으므로 셀을 다시 실행하면
됩니다.

In [3]:
RETRYABLE_HTTP_CODES = {429, 500, 502, 503, 504}
RETRYABLE_NETWORK_ERRORS = (urllib.error.URLError, TimeoutError, ConnectionError, http.client.HTTPException)
_last_request_at = 0.0


def _retry_delay(error, attempt):
    retry_after = getattr(error, "headers", None) and error.headers.get("Retry-After")
    if retry_after:
        try:
            return max(float(retry_after), REQUEST_INTERVAL_SECONDS)
        except ValueError:
            pass
    return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)


def fetch_citation_batch(paper_ids):
    """arXiv id 리스트 순서에 맞춰 Semantic Scholar 응답 리스트(못 찾으면 None)를 반환합니다."""
    global _last_request_at
    url = f"{S2_API_URL}?fields={urllib.parse.quote(S2_FIELDS, safe=',.')}"
    body = json.dumps({"ids": [f"ARXIV:{pid}" for pid in paper_ids]}).encode("utf-8")
    headers = {"Content-Type": "application/json"}
    if S2_API_KEY:
        headers["x-api-key"] = S2_API_KEY
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        request = urllib.request.Request(url, data=body, headers=headers, method="POST")
        try:
            _last_request_at = time.monotonic()
            with urllib.request.urlopen(request, timeout=60, context=ssl.create_default_context()) as response:
                return json.loads(response.read())
        except urllib.error.HTTPError as error:
            if error.code not in RETRYABLE_HTTP_CODES or attempt >= MAX_RETRIES:
                detail = error.read().decode("utf-8", errors="replace")
                error.close()
                raise RuntimeError(f"Semantic Scholar API 오류 {error.code}: {detail[:300]}") from error
            delay = _retry_delay(error, attempt)
            print(f"HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})")
            error.close()
            time.sleep(delay)
        except RETRYABLE_NETWORK_ERRORS as error:
            if attempt >= MAX_RETRIES:
                raise
            delay = min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)
            print(f"네트워크 오류({error!r}): {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})")
            time.sleep(delay)
    raise RuntimeError("재시도 횟수를 초과했습니다.")


def _neighbor(item):
    external = item.get("externalIds") or {}
    return {
        "semantic_scholar_id": item.get("paperId"),
        "arxiv_id": external.get("ArXiv"),
        "title": item.get("title"),
    }


def to_record(paper_id, entry, fetched_at):
    if entry is None:
        return {
            "arxiv_id": paper_id, "found": False, "semantic_scholar_id": None, "title": None,
            "citation_count": None, "reference_count": None, "influential_citation_count": None,
            "citations": [], "references": [], "fetched_at": fetched_at,
        }
    return {
        "arxiv_id": paper_id,
        "found": True,
        "semantic_scholar_id": entry.get("paperId"),
        "title": entry.get("title"),
        "citation_count": entry.get("citationCount"),
        "reference_count": entry.get("referenceCount"),
        "influential_citation_count": entry.get("influentialCitationCount"),
        "citations": [_neighbor(c) for c in (entry.get("citations") or []) if c],
        "references": [_neighbor(r) for r in (entry.get("references") or []) if r],
        "fetched_at": fetched_at,
    }

## 3. 수집 실행과 재시작

기존 `data/citations/*.jsonl`을 먼저 읽어 이미 수집된 `arxiv_id` 집합을 만들고, 대상
목록에서 그만큼을 뺀 나머지만 배치로 조회합니다. 배치 하나를 처리할 때마다 바로
`flush_progress()`로 디스크에 씁니다(완료된 청크를 다시 덮어써서, 언제 중단돼도 다음
실행에서 정확히 이어받습니다). 재시도까지 모두 실패해 예외가 나면 그 시점까지 받은
배치는 이미 저장되어 있으므로, 셀을 다시 실행하면 남은 논문부터 이어서 수집합니다.

In [4]:
def load_existing_records():
    records = []
    chunk_index = 1
    while chunk_path(chunk_index).exists():
        with chunk_path(chunk_index).open(encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f"경고: {chunk_path(chunk_index).name}의 손상된 줄 하나를 건너뜁니다.")
        chunk_index += 1
    return records


def write_chunk(chunk_index, chunk_records):
    path = chunk_path(chunk_index)
    with path.open("w", encoding="utf-8") as file:
        for record in chunk_records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")
    return path


def save_state(done_count):
    with STATE_PATH.open("w", encoding="utf-8") as file:
        json.dump({
            "target_signature": target_signature,
            "target_count": len(target_ids),
            "completed_count": done_count,
            "updated_at": datetime.now(timezone.utc).isoformat(),
        }, file, ensure_ascii=False)


records = load_existing_records()
seen_ids = {r["arxiv_id"] for r in records if r.get("arxiv_id")}
todo = [pid for pid in target_ids if pid not in seen_ids]

if STATE_PATH.exists():
    with STATE_PATH.open(encoding="utf-8") as file:
        old_state = json.load(file)
    if old_state.get("target_signature") != target_signature:
        print("참고: data/cleaned의 논문 목록이 이전 실행과 달라졌습니다. "
              "기존에 모은 인용 정보는 유지하고 늘어나거나 바뀐 논문만 추가로 수집합니다.")

print(f"이미 수집됨: {len(seen_ids)} / 이번에 수집할 논문: {len(todo)} / 배치당 최대 {S2_BATCH_SIZE}편")

next_chunk_index = 1


def flush_progress():
    global next_chunk_index
    completed_chunks = len(records) // CHUNK_SIZE
    while next_chunk_index <= completed_chunks:
        chunk_start = (next_chunk_index - 1) * CHUNK_SIZE
        chunk_end = next_chunk_index * CHUNK_SIZE
        saved_path = write_chunk(next_chunk_index, records[chunk_start:chunk_end])
        print(f"  -> {saved_path.name} 저장 ({chunk_end - chunk_start}건)")
        next_chunk_index += 1
    remainder_start = (next_chunk_index - 1) * CHUNK_SIZE
    if remainder_start < len(records):
        write_chunk(next_chunk_index, records[remainder_start:])


for batch_start in range(0, len(todo), S2_BATCH_SIZE):
    batch_ids = todo[batch_start:batch_start + S2_BATCH_SIZE]
    entries = fetch_citation_batch(batch_ids)
    fetched_at = datetime.now(timezone.utc).isoformat()
    found_in_batch = 0
    for paper_id, entry in zip(batch_ids, entries):
        record = to_record(paper_id, entry, fetched_at)
        records.append(record)
        found_in_batch += int(record["found"])
    flush_progress()
    save_state(len(records))
    print(f"[{len(records)}/{len(target_ids)}] 배치 처리 완료 "
          f"({len(batch_ids)}편 조회, Semantic Scholar에서 찾음 {found_in_batch}편)")

flush_progress()  # 이번 실행에서 새로 처리한 논문이 없어도 마지막 상태를 디스크와 맞춥니다.
save_state(len(records))
saved_files = sorted(OUTPUT_DIR.glob(f"{FILE_PREFIX}_part*.jsonl"))
print(f"완료: 누적 {len(records)}/{len(target_ids)}편, 파일 {len(saved_files)}개")

이미 수집됨: 0 / 이번에 수집할 논문: 6467 / 배치당 최대 500편
[500/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 500편)
HTTP 429: 5초 후 재시도 (1/6)
[1000/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 500편)
[1500/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 500편)
HTTP 429: 5초 후 재시도 (1/6)
HTTP 429: 10초 후 재시도 (2/6)
HTTP 429: 20초 후 재시도 (3/6)
[2000/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 500편)
[2500/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 499편)
[3000/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 498편)
[3500/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 500편)
HTTP 429: 5초 후 재시도 (1/6)
HTTP 429: 10초 후 재시도 (2/6)
HTTP 429: 20초 후 재시도 (3/6)
HTTP 429: 40초 후 재시도 (4/6)
HTTP 429: 80초 후 재시도 (5/6)
HTTP 429: 120초 후 재시도 (6/6)
[4000/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 497편)
[4500/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 498편)
  -> arxiv_citations_part1.jsonl 저장 (5000건)
[5000/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 498편)
[5500/6467] 배치 처리 완료 (500편 조회, Semantic Scholar에서 찾음 500편)
[6000/6

## 4. 저장 결과 검증

In [5]:
saved_records = []
chunk_index = 1
while chunk_path(chunk_index).exists():
    with chunk_path(chunk_index).open(encoding="utf-8") as file:
        saved_records.extend(json.loads(line) for line in file if line.strip())
    chunk_index += 1

saved_ids = [r["arxiv_id"] for r in saved_records]
missing = sorted(set(target_ids) - set(saved_ids))

assert len(saved_ids) == len(set(saved_ids)), "중복된 arxiv_id가 있습니다."

print(f"레코드 수: {len(saved_records)} / 대상 논문 수: {len(target_ids)}")
if missing:
    print(f"아직 수집되지 않은 논문이 {len(missing)}건 남아 있습니다. 위 수집 셀을 다시 실행하면 이어서 처리됩니다.")
else:
    print("대상 논문 전체의 인용 정보를 모두 수집했습니다.")

found_count = sum(1 for r in saved_records if r.get("found"))
print(f"Semantic Scholar에서 찾음: {found_count} / 못 찾음: {len(saved_records) - found_count}")
display(saved_records[:2])

레코드 수: 6467 / 대상 논문 수: 6467
대상 논문 전체의 인용 정보를 모두 수집했습니다.
Semantic Scholar에서 찾음: 6367 / 못 찾음: 100


[{'arxiv_id': '1304.3111',
  'found': True,
  'semantic_scholar_id': '59591020ca3442a46dfaa233c2488450c2d0560a',
  'title': 'Estimating uncertain spatial relationships in robotics',
  'citation_count': 1980,
  'reference_count': 14,
  'influential_citation_count': 120,
  'citations': [{'semantic_scholar_id': '3b8a4599b0c4b1ea89ed6e193bd97b5712b2cbae',
    'arxiv_id': None,
    'title': 'Efficient Distributed Pedestrian Inertial SLAM for Indoor Multi-Pedestrian Localization'},
   {'semantic_scholar_id': '7e14446bb173cf1e227fc7c69cbc15f2cd9f9f6f',
    'arxiv_id': None,
    'title': 'Mobile Robot Localization and SLAM: A Critical Review of Sensors, Multi-Sensor Fusion, and Neural Representations'},
   {'semantic_scholar_id': '0ef4cf914ee9c459dc10a54d205ae2ba0003539d',
    'arxiv_id': '2607.17699',
    'title': 'SLAM in Low-Light Environments: Project Report'},
   {'semantic_scholar_id': 'ddeba9ba30bde64d1ce81d9b4140c82006fa3a51',
    'arxiv_id': None,
    'title': 'Based on Genetic Adapti